In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

import hippynn
from hippynn import active_directory
from hippynn.experiment.serialization import load_model_from_cwd 
from hippynn.graphs.predictor import Predictor
from hippynn.graphs.nodes.physics import GradientNode

from repulsive_potential import RepulsivePotential, RepulsivePotentialBySpeciesNode

hippynn.settings.WARN_LOW_DISTANCES = False

In [ ]:
aa_data_src = "../../../datasets/cg_methanol_trajectory.npz"
model_src = "model"
cg_data_src = "md_results/hippynn_cg_trajectory.npz"

In [ ]:
with active_directory(model_src):
    model = load_model_from_cwd()

sys_energy_node = model.node_from_name("sys_energy") # total system energy

rp_node = model.node_from_name("repulse").mol_energy # energy contribution from repulsive potential
positions_node = model.node_from_name("positions")

rp_force = GradientNode(name="rp_force", parents=(rp_node, positions_node), sign=-1) # force induced by repulsive potential

model = Predictor.from_graph(model, additional_outputs=[sys_energy_node, rp_node, rp_force])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
with np.load(aa_data_src) as f:
    aa_rdf_bins = f['rdf_bins']
    aa_rdf_values = f['rdf_values']

In [ ]:
# You can comment this and some additional code which uses it out if you have not run CG MD and just want to compare
# your repulsive potential with the AA RDF data that was used to create it

with np.load(cg_data_src) as f:
    cg_rdf_bins = f['rdf_bins']
    cg_rdf_values = f['rdf_values']

In [ ]:
# Compute model outputs for a pair of particles at distance x for values of x ranging from 0.1 to 14
x = np.linspace(0.1, 14, 200)
positions = [torch.tensor([[[0,0,0],[xx,0,0]]], dtype=torch.float, device=device) for xx in x]
cells=100*torch.eye(3, dtype=torch.float, device=device).unsqueeze(0)
species=torch.tensor([[1, 1]], dtype=torch.int, device=device)

model_outputs = [model(positions=positions_i, cells=cells, species=species) for positions_i in positions]

In [ ]:
# Plot energy against RDFs
rp_y = [output['repulse.mol_energy'].squeeze() for output in model_outputs]
e_y = [output['sys_energy'].squeeze() for output in model_outputs]

fig, ax1 = plt.subplots(figsize=(10,4))

# Plot energy
color = '#1f77b4'
ax1.plot(x, rp_y, color=color, label='repulsive potential energy contribution', linestyle='--')
ax1.set_xlabel(r'distance ($\mathrm{\AA}$)')
ax1.set_ylabel('energy (kcal/mol)', color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(0, 25)
ax1.set_xlim(0, 14)

ax1.plot(x, e_y, color=color, label='total system energy', alpha=0.5)

# Create a second y-axis that shares the same x-axis
ax2 = ax1.twinx()

# Plot RDFs
color ='#ff7f0e'
ax2.plot(aa_rdf_bins, aa_rdf_values, color=color, alpha=0.5, linestyle='--', label='AA RDF')
ax2.set_ylabel('g(x) (-)', color=color)
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(0, 2)

ax2.plot(cg_rdf_bins, cg_rdf_values, color=color, label='CG RDF')

# Add legends
fig.legend(loc="upper right", bbox_to_anchor=(0.87, 0.49))

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
# Plot force against RDFs
rp_y = [output['rp_force'] for output in model_outputs]
rp_y = [(output[0,0]**2).sum()**(1/2) for output in rp_y]
f_y = [output['forces'] for output in model_outputs]
f_y = [(output[0,0]**2).sum()**(1/2) for output in f_y]

fig, ax1 = plt.subplots(figsize=(10,4))

# Plot energy
color = '#1f77b4'
ax1.plot(x, rp_y, color=color, label='repulsive potential force contribution', linestyle='--')
ax1.set_xlabel(r'distance ($\mathrm{\AA}$)')
ax1.set_ylabel(r'force magnitude (kcal/(mol$\cdot\mathrm{\AA}$))', color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(0, 15)
ax1.set_xlim(0, 14)

ax1.plot(x, f_y, color=color, label='total force', alpha=0.5)

# Create a second y-axis that shares the same x-axis
ax2 = ax1.twinx()

# Plot RDFs
color ='#ff7f0e'
ax2.plot(aa_rdf_bins, aa_rdf_values, color=color, alpha=0.5, linestyle='--', label='AA RDF')
ax2.set_ylabel('g(x) (-)', color=color)
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(0,2)

ax2.plot(cg_rdf_bins, cg_rdf_values, color=color, label='CG RDF')

# Add legends
fig.legend(loc="upper right", bbox_to_anchor=(0.87, 0.9))

# Show the plot
plt.tight_layout()
plt.show()